# A categorically organized CAS in Lean

This notebook is the acceptance proof of the CasDsl vertical slice: a
computer algebra system whose **user-facing interfaces are organized by the
mathematical categories where operations first make sense** — not by
implementation classes, and not by backends.

Everything below runs in a persistent Lean 4 kernel
([lean-jupyter-kernel](https://github.com/dzackgarza/lean-jupyter-kernel)).
Three invariants to watch for:

1. **Backend-blind syntax.** You will never see a backend named in an
   expression. Some results below are computed by SageMath through a direct
   typed adapter — the *developer's* routing configuration decides that,
   and `#explain_route` will show it. The mathematics doesn't change.
2. **Category-owned methods.** `factor`, `det`, `annihilator`, `nth` are
   declared on categories; objects receive them by membership and by
   *subcategory inheritance*, never by forwarding code on a leaf class.
3. **Semantic availability ≠ computability.** A method that makes
   mathematical sense stays available even when no implementation route
   exists yet — execution then fails with a *structured capability gap*
   (an auditable developer backlog item), never a fake value and never a
   type error. The final cell demonstrates this deliberately.

4. **LaTeX-first results.** A result with a natural LaTeX form is typeset
   automatically, with no `show()` call: the cell publishes it as
   `text/latex` beside the plain text, which stays in the bundle as the
   fallback. A value with no natural LaTeX form — a truth value, in §10 —
   displays that plain text alone.


## 1 · Trusted arithmetic and assertions

`assert` is an *operational* assertion in the ordinary CAS sense: the
predicate is computed and trusted, with a fourfold outcome
`true | false | unknown | error`. Only `true` lets the cell commit.
No Lean theorem is generated, and no certificate is required — this is a
CAS, not a proof obligation machine.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/home/dzack/gitclones/lean-cas-dsl)…


1:0: ✓ 2 + 3 = 5


In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Backend-blind factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by **subcategory inheritance through the category
graph**, and is executed by whatever implementation the developer routed.


In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

The expression above never mentioned a backend. The routing that chose one
is developer diagnostics, not mathematics:


In [5]:
#explain_route n.factor()

1:0:   method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ


  method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ

`gcd` is declared on the same category as `factor` — greatest common
divisors exist in every unique factorization domain, which is where the
operation first makes sense. SPEC.md writes it in *prefix* position, and
that is exactly what it is: `gcd(84, 30)` **is** the method call
`84.gcd(30)`, resolved and routed identically. The prefix reading applies
only to a name that is unbound *and* declared as a method somewhere, so it
can never shadow a binding or invent an operation.

In [6]:
assert gcd(84, 30) = 6

1:0: ✓ gcd(84, 30) = 6


## 3 · Polynomials, canonical maps, and calling a polynomial

`ℤ ⊆ ℚ` denotes the preferred canonical map — so `map p to ℚ[x]` moves a
polynomial along it without ceremony. `factor` is routed where `p` lives:
ℤ[x] is a UFD, and comparing its factorization with the one in ℚ[x]
below — content and units differ in general — is itself instructive. And a polynomial can simply be
**called**: elaboration inserts evaluation through the preferred compatible
coefficient map. The mathematician writes `q(1)`, as on paper.

`map e to D` means: apply the preferred canonical map into `D` when one is
registered, and fail honestly otherwise. Canonical maps are *preferred
choices*, not necessarily injections — the inclusions `ℕ ⊆ ℤ ⊆ ℚ` are
monomorphisms, while `ℤ → ℤ/n` is the ring quotient, supplied by its
universal property. As of round two these are **registry data**: the
prelude registers them, the engine knows none of these facts, and an
unregistered pair fails with the honest `there is no preferred canonical
map` error — a missing coercion is never widened to a "reasonable"
conversion.

In [7]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


Two membership questions about the ring `p` was just defined in. The second
is ordinary — `p` is a value, `ℤ[x]` is a set. The first is the interesting
one: `x` is *not* a session binding (defining `p` published nothing), but in
`x ∈ ℤ[x]` the only element of that ring a repeated `x` could name is its
indeterminate, so the assertion reads it that way — **locally**, for this
one line. Outside it, a bare `x` is still the honest `not bound` error.

In [8]:
assert x ∈ ℤ[x]

1:0: ✓ x ∈ ℤ[x]


In [9]:
assert p ∈ ℤ[x]

1:0: ✓ p ∈ ℤ[x]


`deg` is declared on `PolynomialElems`, a category `p` inhabits
*independently* of the factorization hierarchy: degree is a structural read
of a polynomial and makes sense over coefficient rings where factorization
does not. Neither membership implies the other.

In [10]:
p.deg()

1:0: 3


3

In [11]:
p.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [12]:
let q := map p to ℚ[x]

1:0: q := x^3 - 2x + 1 ∈ ℚ[x]


In [13]:
q.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [14]:
assert q(1) = 0

1:0: ✓ q(1) = 0


`roots` answers in the polynomial's **own coefficient ring** — that is the
mathematical content of the method, not a limitation of the backend. Over
ℤ, `x³ − 2x + 1` has the root `1`:

In [15]:
p.roots()

1:0: {1}


{1}

In [16]:
assert 1 ∈ p.roots()

1:0: ✓ 1 ∈ p.roots()


And SPEC.md's `x² − 2` over ℚ has **none**. The empty set is the answer —
not an error, and not a silent reach into an extension field. The roots that
polynomial does have live in ℂ, one explicit `map` away — §12.

In [17]:
let q := x ↦ x² - 2 in ℚ[x]

1:0: q := x^2 - 2 ∈ ℚ[x]


In [18]:
q.roots()

1:0: {}


{}

The quotient `ℤ → ℤ/n` is one registered rule for *every* modulus: an
integer names its residue class. `n` is still `360`, and `360 ≡ 3 (mod 7)`:

In [19]:
map n to ℤ/7

1:0: 3


3

The coercion surface is itself auditable: these are the *only* maps
the surface will ever insert, straight from the registry (#9):

In [20]:
#canonical_maps

1:0:   map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts
  ℕ → ℝ       identity
              ℕ ⊆ ℝ: a natural number is a real number
  ℤ → ℝ       identity
              ℤ ⊆ ℝ: an integer is a real number
  ℚ → ℝ       identity
              ℚ ⊆ ℝ: SPEC.md's own chain link — every rational IS a real, and the value presenting it does not change
  ℕ → ℂ       id

  map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts
  ℕ → ℝ       identity
              ℕ ⊆ ℝ: a natural number is a real number
  ℤ → ℝ       identity
              ℤ ⊆ ℝ: an integer is a real number
  ℚ → ℝ       identity
              ℚ ⊆ ℝ: SPEC.md's own chain link — every rational IS a real, and the value presenting it does not change
  ℕ → ℂ       identit

## 4 · Exact matrix algebra

Matrix literals use row-semicolon syntax; `det` and `inverse` are methods
of the square-matrix category, computed exactly over `ℚ`.


In [21]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

1:0: M := [1, 2; 3, 4] ∈ Mat₂(ℚ)


In [22]:
M.inverse()

1:0: [-2, 1; 3/2, -1/2]


[-2, 1; 3/2, -1/2]

In [23]:
assert M.det() = -2

1:0: ✓ M.det() = -2


A **vector is its own kind of object.** `Matₙ` is square by construction, so a
column of length 2 is not a 2×2 thing, and keeping them apart is exactly what
lets the application below be *shape-checked* rather than silently
reinterpreted. `SPEC.md` writes the action four ways — they are one operation.

In [24]:
let v := (1, 2) in ℚ²

1:0: v := (1, 2) ∈ ℚ²


In [25]:
let b := (5, 11) in ℚ²

1:0: b := (5, 11) ∈ ℚ²


In [26]:
assert M*v = b

1:0: ✓ M*v = b


In [27]:
assert M⁻¹ b = v

1:0: ✓ M⁻¹ b = v


In [28]:
assert M⁻¹(M v) = v

1:0: ✓ M⁻¹(M v) = v


In [29]:
assert M(M⁻¹ b) = b

1:0: ✓ M(M⁻¹ b) = b


`⁻¹` is the spelling of the `inverse` **method**, not a second operation, and
juxtaposition is the product. A vector of the wrong length is a loud refusal
naming both shapes:

In [30]:
M*(1, 0, 1)

LeanError: Mat₂(ℚ) does not apply to a vector of length 3: a matrix applies to vectors of its own size, and these shapes do not meet

The **rank** and the **kernel** are two reads of one reduced row echelon form,
so they are exact and native — no backend is asked, and none can lie. The
kernel is a *subspace*, presented by a basis; an invertible matrix kills
nothing, so this one is the trivial subspace, which is the set `{0}`.

In [31]:
assert M.rank() = 2

1:0: ✓ M.rank() = 2


In [32]:
M.ker()

1:0: span_ℚ{} ≤ ℚ²


span_ℚ{} ≤ ℚ²

In [33]:
assert M.ker() = {0}

1:0: ✓ M.ker() = {0}


In [34]:
assert M.trace() = 5

1:0: ✓ M.trace() = 5


## 5 · Subcategory inheritance, for real

`annihilator : Modules(ℤ) → Ideals(ℤ)` is declared **once**, on the parent
category. `F` below is declared in the *proper subcategory*
`SmallModules(ℤ)` — which contains **no forwarding declaration**. The
method arrives purely through the registered inclusion
`SmallModules ≤ Modules`. (The ascription is doing real semantic work:
`ℤ/4` *in a module category* means the ℤ-module ℤ/4, not the ring.)


In [35]:
let F := ℤ/4 in SmallModules(ℤ)

1:0: F := ℤ/4 as ℤ-module


In [36]:
F.annihilator()

1:0: (4)


(4)

## 6 · Transport along preferred functors

`cardinality` is declared on `Sets` — and a module is not a set. But the
prelude registers the forgetful functor `UnderlyingSet : Modules(ℤ) → Sets`
as *preferred*, so the resolver transports the **receiver**:
`F.cardinality()` resolves as `UnderlyingSet(F).cardinality()`. Nothing was
declared on modules, no forwarding method exists anywhere, and the ordinary
call syntax is unchanged.

In [37]:
F.cardinality()

1:0: 4


4

Membership transports the same way — `2` names a residue class of the
underlying set:

In [38]:
assert 2 ∈ F

1:0: ✓ 2 ∈ F


Equality, by contrast, is **category-bound**. `U(F) = {0, 1, 2, 3}` in
Sets — but `F` itself is a module, and there is no *unique* module
structure on that set, so bare `=` between objects of different
categories is trivially false: it never inserts the functor. Comparing
them requires explicitly asking the question in a common comparison
category — which is exactly what the Sets method `set_eq` does (its
receiver transports, like `∈` above):

In [39]:
assert F ≠ {0, 1, 2, 3}

1:0: ✓ F ≠ {0, 1, 2, 3}


In [40]:
F.set_eq({0, 1, 2, 3})

1:0: true


true

Two guarantees, both machine-checked in the build:

- transport runs **only where direct resolution finds nothing** —
  `annihilator` above still arrives untransported through
  `SmallModules(ℤ) ≤ Modules(ℤ)`, so registering a functor can never take
  a method away from an object that already had it;
- two applicable functors would be an honest *ambiguity error* naming both,
  never a silent pick.

The transport step itself is developer diagnostics, not mathematics:

In [41]:
#explain_route F.cardinality()

1:0:   method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set


  method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set

## 7 · Countable sets, ellipses, and indexing

Countability is mathematical structure — a monomorphism into ℕ — not a
backend capability. A *registered enumeration choice* labels elements, so
countable objects support `nth` (`X[k]`, 0-based) and `cardinality`.
Ellipsis literals are the exact Haskell-style progressions, nothing more.

The registered convention for `ℤ` is `0, 1, −1, 2, −2, …` — a documented,
revisitable choice, never a claim that ℤ is intrinsically ordered that way.


In [42]:
let X := {0, 1, 2, ...}

1:0: X := {0, 1, ...}


In [43]:
assert X = ℕ

1:0: ✓ X = ℕ


In [44]:
let Y := {0, 2, 4, ...}

1:0: Y := {0, 2, ...}


In [45]:
assert 8 ∈ Y

1:0: ✓ 8 ∈ Y


In [46]:
assert 9 ∉ Y

1:0: ✓ 9 ∉ Y


In [47]:
ℤ[3]

1:0: 2


2

`ℚ` indexes by *its* registered convention too — the Cantor zigzag
(`0, 1, −1, 1/2, −1/2, 2, −2, 1/3, …`, reduced fractions only), a
documented revisitable choice exactly like ℤ's (round three, #17):

In [48]:
ℚ[3]

1:0: 1/2


1/2

In [49]:
X.cardinality()

1:0: ℵ₀


ℵ₀

## 8 · Functions

A function is `binder ↦ body` together with the domains it runs between, and
the two spellings SPEC.md uses — the lambda and `f(t) := …` — denote the
*same* function. `ℝ → ℝ` is an ascription **domain tag** at this stage: it
says where the function is declared and attaches no analysis semantics.

Bodies are exact polynomials, which is what lets the assertions below be
identities of function *expressions* rather than samples at a few points.
A body the polynomial engine cannot express (`t ↦ sin(t)`, `t ↦ e^t`) used to
be an honest gap here; it is now read **symbolically** instead — see the
calculus section, where such a body is presented so a limit, a definite
integral or a Taylor expansion can be taken of it. The polynomial reading is
still preferred wherever it applies, because it is the one that *decides*.

In [50]:
let h := t ↦ t² + 1 in ℝ → ℝ

1:0: h := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [51]:
let hp(t) := t^2 + 1 in R->R

1:0: hp := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [52]:
assert h = hp

1:0: ✓ h = hp


In [53]:
assert h(0) = 1

1:0: ✓ h(0) = 1


In [54]:
assert h(3) = 10

1:0: ✓ h(3) = 10


`h(-t) = h(t)` is not two numeric samples that happened to agree: both
sides *substitute* a polynomial into the body, and the normal forms are
compared. A function in scope makes the binder it names available as that
indeterminate, which is how `t` can be written freely here.

In [55]:
assert h(-t) = h(t)

1:0: ✓ h(-t) = h(t)


The leading-ascription spelling declares the same thing. `e` returns in §10,
where its image is a set.

In [56]:
let e: ℕ → ℕ := n ↦ 2n

1:0: e := n ↦ 2n ∈ ℕ → ℕ


Composition is the point of having functions at all: `f ∘ g` is a function
like any other, and the identity it satisfies is decided by substituting one
body into the other.

In [57]:
let f(t) = t^2 in RR->RR

1:0: f := t ↦ t^2 ∈ ℝ → ℝ


In [58]:
let g(t) = t^3 in RR->RR

1:0: g := t ↦ t^3 ∈ ℝ → ℝ


In [59]:
assert (f ∘ g)(t) = t^6

1:0: ✓ (f ∘ g)(t) = t^6


## 9 · Finite sets and set algebra

The set operations are *category-owned methods on `Sets`*, exactly like
`cardinality` and `contains`: `A ∪ B` **is** `A.union(B)`, resolved and
routed like any other call. The ascription says where the set lives —
`𝒫(ℤ)` and `2^ℤ` are the two spellings SPEC.md uses for the same powerset,
and ascribing to one is a *checked membership judgment* (`A ⊆ ℤ`), not an
annotation.

In [60]:
let A := {1, 2, 3} in 𝒫(ℤ)

1:0: A := {1, 2, 3}


In [61]:
let B := {3, 4, 5} in 2^ℤ

1:0: B := {3, 4, 5}


In [62]:
assert A ∪ B = {1, 2, 3, 4, 5}

1:0: ✓ A ∪ B = {1, 2, 3, 4, 5}


In [63]:
assert A ∩ B = {3}

1:0: ✓ A ∩ B = {3}


In [64]:
assert A \ B = {1, 2}

1:0: ✓ A \ B = {1, 2}


In [65]:
assert A △ B = {1, 2, 4, 5}

1:0: ✓ A △ B = {1, 2, 4, 5}


`|A|` is the cardinality method under another spelling. The interesting
cases are the two sets that are **denoted rather than listed**: the elements
of `A × B` are pairs and the elements of `𝒫(A)` are sets, and this slice's
`Value` presents neither — so both are *presentations*, like `{0, 2, 4, ...}`
is. Their cardinalities are exact cardinal arithmetic, which is what makes
`|𝒫(A)| = 2^|A|` a computed identity rather than a definition.

In [66]:
|A|

1:0: 3


3

In [67]:
A × B

1:0: {1, 2, 3} × {3, 4, 5}


{1, 2, 3} × {3, 4, 5}

In [68]:
assert |A × B| = 9

1:0: ✓ |A × B| = 9


In [69]:
assert |𝒫(A)| = 2^|A|

1:0: ✓ |𝒫(A)| = 2^|A|


Membership and inclusion are one decision procedure with two spellings:
`X ∈ 𝒫(A)` asks exactly what `X ⊆ A` asks. (SPEC.md also writes the ASCII
`in` for `∈`.)

The routes tell the usual second story. `∪ ∩ \ △` are *meaningful* for every
set — they are declared on `Sets` — but only explicit finite receivers are
routed, so `ℤ ∪ A` resolves and then reports a structured gap. `𝒫(ℕ)` is
uncountable, and this slice's `Cardinality` says it cannot state that size
rather than inventing `ℵ₀`. Both appear in the audit at the end.

In [70]:
assert 2 ∈ A

1:0: ✓ 2 ∈ A


In [71]:
assert 4 ∉ A

1:0: ✓ 4 ∉ A


In [72]:
assert A ⊆ A ∪ B

1:0: ✓ A ⊆ A ∪ B


In [73]:
assert A in 𝒫(ℤ)

1:0: ✓ A ∈ 𝒫(ℤ)


## 10 · Set comprehensions

`{n ∈ ℤ | n² ≤ 20}` is a claim about *all* integers, so it is **decided**,
never sampled. The guard is rewritten as `p(n) ⋈ 0` for an exact polynomial;
a Cauchy-style bound puts every root of `p` inside `±N`, so `p` keeps one
sign on each tail; evaluating it there says whether the tail satisfies the
guard. A satisfied tail means the set is infinite and the comprehension says
so — otherwise every solution lies inside the bound and each candidate is
tested exactly. There is no enumeration cutoff anywhere in that.

In [74]:
let S := {n ∈ ℤ | n² ≤ 20}

1:0: S := {-4, -3, -2, -1, 0, 1, 2, 3, 4}


In [75]:
assert S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}

1:0: ✓ S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}


In [76]:
assert |S| = 9

1:0: ✓ |S| = 9


An infinite comprehension is presented when its image *is* a presentation
the slice already has. `{2n | n ∈ ℕ}` is the arithmetic progression
`{0, 2, 4, ...}` — SPEC.md's own identity — so membership in it is **solved**
rather than searched: `10¹² ∈ E` costs exactly what `8 ∈ E` costs.

In [77]:
let E := {2n | n ∈ ℕ}

1:0: E := {0, 2, ...}


In [78]:
assert 8 ∈ E

1:0: ✓ 8 ∈ E


In [79]:
assert 9 ∉ E

1:0: ✓ 9 ∉ E


In [80]:
assert |E| = ℵ₀

1:0: ✓ |E| = ℵ₀


In [81]:
assert 1000000000000 ∈ E

1:0: ✓ 1000000000000 ∈ E


`e` was declared in §8 as `n ↦ 2n`. Its **image** is that same set, under
either spelling: `e.image()` is the one method functions own, and `e(ℕ)` —
applying a function to its source — is the same call. A guard bounds the
binder to a finite range, which is what turns the last SPEC.md line into an
explicit list.

In [82]:
assert e(ℕ) = E

1:0: ✓ e(ℕ) = E


In [83]:
e.image()

1:0: {0, 2, ...}


{0, 2, ...}

In [84]:
{e(n) | n ∈ ℕ, 0 ≤ n < 6}

1:0: {0, 2, 4, 6, 8, 10}


{0, 2, 4, 6, 8, 10}

**Disclosed gaps — the two SPEC.md §Ellipses comprehension lines, which fail
differently.**

`{n in ℕ | n.is_prime()}` *parses* and is refused **at the binding**: this
slice decides comprehensions whose guard is a polynomial comparison in the
binder, and a primality test is not one. No sampled membership, no truncated
enumeration, no wrong verdict — and the method itself works fine outside a
comprehension, as the cell below shows.

`{n in ℕ | f(n) ∈ 2ℕ}` does not even parse: `∈` is an assertion relation, not
a term operator, so the guard position rejects it and the statement splitter
runs what is left as fragments. That reading is the splitter's, not this
surface's: a decided answer to a *different* claim, which is exactly why the
line is listed as a gap rather than trusted.

`2ℕ` no longer compounds it. There is still no scaling of a set by a number,
but implicit multiplication takes an *atom* since #26, so `2ℕ` is the product
`2 · ℕ` and **refuses loudly** — `ℕ` is not an element value. It used to split
into two statements and quietly assert `Y = 2` while printing `ℕ` beside it,
which answered a claim nobody made.

In [85]:
let m7 := 7 in ℤ

1:0: m7 := 7 ∈ ℤ


In [86]:
m7.is_prime()

1:0: true


true

## 11 · Exact number systems

`SPEC.md` opens with a chain of inclusions. It is not a set-theoretic
computation and it is not hard-coded either: which domains include which is
the **canonical-map registry's** claim — the same registry `map p to ℚ[x]`
consulted in §3 — and `⊆` between two domains means *the preferred canonical
map exists and is an inclusion*.

In [87]:
assert ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ

1:0: ✓ ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ


So ℝ and ℂ have **inhabitants**: exact algebraic numbers, written `a + b√d`.
`√2` is not 1.4142135…; it is the number whose square is exactly 2, and the
next cell is a membership judgment about it rather than a rounding.

In [88]:
assert √2 ∈ ℝ

1:0: ✓ √2 ∈ ℝ


In [89]:
assert √2 ∉ ℚ

1:0: ✓ √2 ∉ ℚ


In [90]:
assert 2 + 2i ∈ ℂ

1:0: ✓ 2 + 2i ∈ ℂ


`i` is a constant, not a binding — and a `let i := …` would shadow it, exactly
as `let R := …` shadows the ASCII spelling of ℝ.

The four operations of the complex plane are `re`, `im`, `bar` and `|·|`.
They are ordinary category methods (on `ComplexElems`), and they are *native*:
reading the real part of `a + b√d` is something the engine genuinely decides,
so no backend is asked.

In [91]:
let z := 2 + 2i in ℂ

1:0: z := 2 + 2i ∈ ℂ


In [92]:
z.bar()

1:0: 2 - 2i


2 - 2i

In [93]:
assert z · z.bar() = 8

1:0: ✓ z · z.bar() = 8


The modulus is where exactness earns its keep. `|2 + 2i|` is `√8`, and `√8`
**is** `2√2` — one value with one normal form, so the assertion below is an
identity rather than a comparison of decimals.

In [94]:
|z|

1:0: 2√2


2√2

In [95]:
assert |z| = 2√2

1:0: ✓ |z| = 2√2


The bars are one spelling of two methods: `cardinality` for a set, `abs` for
an element. A receiver that is neither gets the ordinary "not a method of any
category this object belongs to" error — the bars invent no third notion of
size.

One square root over ℚ is the **ceiling**, and it is stated rather than
approximated: the next cell *fails on purpose*.

In [96]:
√2 + √5

LeanError: the sum of √2 and √5 leaves the exact form a + b√d this slice presents (one square root over ℚ, and both operands in it): that is a gap, not an approximation

## 12 · ℂ[x], where the cubic splits

`p = x³ − 2x + 1` factored in §3 over ℤ and over ℚ, both times into a linear
factor and an irreducible quadratic. Over ℂ it splits completely — and the
coefficients that appear are exact algebraic numbers, not decimals.

`map p to ℂ[x]` is the ordinary registered coercion (ℤ ⊆ ℂ, applied
coefficient-wise), and `factor` is routed for ℂ[x] like any other ring.

In [97]:
let pc := map p to ℂ[x]

1:0: pc := x^3 - 2x + 1 ∈ ℂ[x]


In [98]:
pc.factor()

1:0: (x - 1) * (x + 1/2 - (1/2)√5) * (x + 1/2 + (1/2)√5)


(x - 1) * (x + 1/2 - (1/2)√5) * (x + 1/2 + (1/2)√5)

That display is a claim, so here it is checked: each root it shows is a root.

In [99]:
assert pc((-1 + √5) / 2) = 0

1:0: ✓ pc((-1 + √5) / 2) = 0


In [100]:
assert pc((-1 - √5) / 2) = 0

1:0: ✓ pc((-1 - √5) / 2) = 0


`roots` answers in the polynomial's **own coefficient ring** — the decision
§3 made when `x² − 2` over ℚ returned the empty set. That decision stands,
and it is why the reach into ℂ is explicit:

In [101]:
q.roots()

1:0: {}


{}

In [102]:
let qc := map q to ℂ[x]

1:0: qc := x^2 - 2 ∈ ℂ[x]


In [103]:
qc.roots()

1:0: {-√2, √2}


{-√2, √2}

`ℂ - ℚ` is a **denoted** set, like `A × B` and `𝒫(A)`: its membership is
decided pointwise, and everything that would need an element list of it
refuses.

`SPEC.md` asserts the inclusion for `q.roots()` itself. Over ℚ that set is
empty, so **the first cell below checks nothing** — it is true the way any
statement about the elements of the empty set is true. The second one is the
claim with content in it. Which of the two `roots` should *default* to is an
open question, recorded in `DESIGN.md` §Open questions and escalated: the
coefficient ring stays the default until it is ruled on, and neither reading
is hidden in the meantime.

In [104]:
assert q.roots() ⊆ ℂ - ℚ

1:0: ✓ q.roots() ⊆ ℂ - ℚ


In [105]:
assert qc.roots() ⊆ ℂ - ℚ

1:0: ✓ qc.roots() ⊆ ℂ - ℚ


## 13 · Numerical approximation

Nothing above produced a decimal. `√2` stayed `√2`, `|2 + 2i|` was `2√2`, and
the cubic's roots came back as exact algebraic numbers — because a decimal is a
different kind of object, and this system makes you say when you want one.

`SPEC.md` says how: numerical approximation is an **operation on an exact
element**, written `map x to ℝ/O(ε)`.

In [106]:
map √2 to ℝ/O(1/10^{10})

1:0: 1.4142135623 + O(1/10^{10})


1.4142135623 + O(1/10^{10})

Those ten digits are the whole claim, and they are **checked rather than
trusted**. The backend computes the decimal — which arbitrary-precision
arithmetic it uses to get there is its own business, and nothing in this
surface names one — and returns the tolerance it achieved alongside it. This
side then verifies `|√2 − 1.4142135623| < 10⁻¹⁰` *exactly*, by squaring, before
the value exists at all. A backend that returned a wrong digit would fail at
that boundary instead of publishing it.

The exact value is **kept, not replaced**: the approximation carries `√2` with
it, and the tolerance the display names is the one you asked for.

`ℝ/O(ε)` is **sugar for a requested tolerance, not a quotient**. `|a − b| < ε`
is not transitive, so there are no classes to speak of: nothing is an element of
`ℝ/O(ε)`, and ℝ is not contained in it. Written anywhere but after `map … to`,
it says exactly that — the next cell *fails on purpose*.

In [107]:
ℝ/O(1/10)

LeanError: `ℝ/O(ε)` is the TARGET of `map … to ℝ/O(ε)` — a requested tolerance for a decimal presentation — and nothing else: it is not a domain, not a set, and not a quotient of ℝ (|a - b| < ε is not transitive, so there are no classes here to be an element of, or to include ℝ in)

Which values may be presented in ℝ at all is the **canonical-map registry's**
question — the same registry behind `map p to ℚ[x]` in §3 — so everything the
⊆-chain of §11 covers can be asked for:

In [108]:
map 1/3 to ℝ/O(1/10^{3})

1:0: 0.333 + O(1/10^{3})


0.333 + O(1/10^{3})

…and ℂ is refused by that registry, because no map of ℂ into ℝ is registered.
Dropping the imaginary part, or quietly answering with the modulus, would be
answering a question nobody asked. *Fails on purpose:*

In [109]:
map 2 + 2i to ℝ/O(1/10^{4})

LeanError: there is no preferred canonical map of 2 + 2i into ℝ

Two more refusals, and they say different things. `O(0)` is not a tolerance at
all — no finite decimal lies within 0 of an irrational number — so that is a
statement about the **request**. A tolerance that no configured backend can
meet is a **capability** failure instead, and it names what was asked for, the
way every capability gap in §14 names its method. Both *fail on purpose*:

In [110]:
map √2 to ℝ/O(0)

LeanError: O(0) is not a tolerance: no finite decimal presentation lies within 0 of an irrational number, so an absolute tolerance is a POSITIVE rational

In [111]:
map √2 to ℝ/O(1/10^{2000})

LeanError: no configured backend produced a decimal presentation within O(1/10^{2000}) — a CAPABILITY failure, not a defect in the value that was asked about:
the 'sage' backend failed (tolerance_not_met): a tolerance of O(1/10^{2000}) needs more than 1000 decimal digits, which is past this adapter's ceiling
  requested tolerance: O(1/10^{2000})

Finally, an approximation has **no arithmetic**. It carries a requested
tolerance, not an error term, and this slice does not invent an error calculus
to propagate one: do the arithmetic exactly and approximate the result. *Fails
on purpose:*

In [112]:
(map √2 to ℝ/O(1/10^{4})) + 1

LeanError: addition is not defined on an approximation (1.4142 + O(1/10^{4})): the value carries a REQUESTED tolerance, not an error term, and this slice does not invent an error calculus to propagate one — compute exactly, then ask for a decimal presentation of the result

## 14 · Subspaces and spans

A subspace is a **set** — its elements are vectors — presented by a basis in
**reduced row echelon form**. That one normal form is a function of the
subspace and of nothing else, which is what makes all four of `SPEC.md`'s
questions decidable from the presentation alone: the dimension is the basis
size, membership is one solved reduction, equality is the bases compared as
data, and inclusion is membership of each basis vector.

`\leq` here means **subobject** (`SPEC.md`'s own note), and the ambient space
is checked rather than taken on trust. `QQ-Mod` is an ascription *tag* at this
stage: the membership it states is real, while the category it names — its
morphisms, its limits, the lattice `≤` really lives in — is deferred to the
categorical foundations this repo deliberately does not have yet.

In [113]:
let u₁ := (1, 0, 1) in ℚ³

1:0: u₁ := (1, 0, 1) ∈ ℚ³


In [114]:
let u₂ := (0, 1, 1) in ℚ³

1:0: u₂ := (0, 1, 1) ∈ ℚ³


In [115]:
let W := span_QQ{u₁, u₂} \leq ℚ³ in QQ-Mod

1:0: W := span_ℚ{(1, 0, 1), (0, 1, 1)} ≤ ℚ³


In [116]:
assert W.dim() = 2

1:0: ✓ W.dim() = 2


In [117]:
assert (1, 1, 2) ∈ W

1:0: ✓ (1, 1, 2) ∈ W


In [118]:
assert (1, 1, 0) ∉ W

1:0: ✓ (1, 1, 0) ∉ W


A dependent generator contributes nothing, and a *different* generating list of
the same subspace is the same subspace — because both reduce to one basis:

In [119]:
span_QQ{u₁, u₂, (1, 1, 2)}

1:0: span_ℚ{(1, 0, 1), (0, 1, 1)} ≤ ℚ³


span_ℚ{(1, 0, 1), (0, 1, 1)} ≤ ℚ³

In [120]:
assert span_QQ{(1, 1, 2), (0, 1, 1)} = W

1:0: ✓ span_QQ{(1, 1, 2), (0, 1, 1)} = W


`SPEC.md` finishes this section with `φ: ℚ³ → ℚ := (a, b, c) ↦ a + b - c` and
`W = ker φ`. Functions here are grounded in the *univariate* polynomial engine,
so a body in three variables is not expressible — and the next cell **fails on
purpose**, naming that as a gap rather than as a syntax error, and saying what
it blocks.

In [121]:
let φ: ℚ³ → ℚ := (a, b, c) ↦ a + b - c

LeanError: a lambda with SEVERAL binders — SPEC.md's `(a, b, c) ↦ a + b - c` — is a disclosed GAP, not a syntax error: functions here are grounded in the univariate polynomial engine, so a body in more than one variable is not expressible and is refused rather than approximated. It is on the SPEC.md ledger (#24), and it is what `W = ker φ` waits on; the span, dimension and membership lines of that section need none of it

## 15 · A composed computation

The last block of `SPEC.md`'s elementary sections puts several of these
together. `{a ∈ ℂ | r(a) = 0}` is the **root set** of an equation — not the
guarded comprehension, which bounds a binder from an *order* comparison, but
the `roots` method the surface already has, reached by reading the equation
with the binder as the indeterminate. The index domain says where the roots are
sought, so `ℂ` is a claim about the splitting rather than a default.

In [122]:
let r(x) := x³ - 2x + 1 in ℚ[x]

1:0: r := x^3 - 2x + 1 ∈ ℚ[x]


In [123]:
let roots := {a ∈ ℂ | r(a) = 0} in 𝒫(ℂ)

1:0: roots := {-1/2 - (1/2)√5, -1/2 + (1/2)√5, 1}


In [124]:
roots

1:0: {-1/2 - (1/2)√5, -1/2 + (1/2)√5, 1}


{-1/2 - (1/2)√5, -1/2 + (1/2)√5, 1}

In [125]:
assert |roots| = 3

1:0: ✓ |roots| = 3


In [126]:
assert 1 ∈ roots

1:0: ✓ 1 ∈ roots


`∑` and `∏` fold a finite set, exactly — over the surds this cubic splits into,
the sum is `0` and not `0.0000001`. (The empty sum is `0` and the empty product
is `1`; a set whose elements have no arithmetic is a refusal naming the value,
never the fold's own seed.)

In [127]:
assert ∑_{a ∈ roots} a = 0

1:0: ✓ ∑_{a ∈ roots} a = 0


In [128]:
assert ∏_{a ∈ roots} a = -1

1:0: ✓ ∏_{a ∈ roots} a = -1


And the **companion matrix** closes the loop: the matrix whose characteristic
polynomial is the one it came from. Which of the four companion *layouts* the
backend uses is its own convention, so what this side checks is what every
layout shares — the size, and the trace.

In [129]:
let C := r.companion_matrix()

1:0: C := [0, 0, -1; 1, 0, 2; 0, 1, 0] ∈ Mat₃(ℚ)


In [130]:
C

1:0: [0, 0, -1; 1, 0, 2; 0, 1, 0]


[0, 0, -1; 1, 0, 2; 0, 1, 0]

In [131]:
assert C.charpoly() = r

1:0: ✓ C.charpoly() = r


In [132]:
assert C.det() = -1

1:0: ✓ C.det() = -1


In [133]:
assert C.trace() = 0

1:0: ✓ C.trace() = 0


## 16 · Semantic availability is not computability

`det` makes sense for a square matrix over any commutative ring — the
category layer says so (`MatrixElems`), and no implementation hole is
allowed to redefine the mathematics. But the developer has only routed
matrices with entries in ℚ: over the field ℤ/5, `det` is *semantically*
available and not yet executable. `gcd` is in the same position outside
ℤ — gcds exist in every UFD, and only the ℤ route is registered. The
audit surface shows every such hole as structured backlog:

In [134]:
#capability_gaps

1:0:   representative    method        category              status
  ℤ                 union         CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 intersect     CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 diff          CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 symdiff       CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℚ                 union   

  representative    method        category              status
  ℤ                 union         CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 intersect     CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 diff          CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℤ                 symdiff       CountableSets(ℤ)      no route matches this presentation (1 registered for the method, none matching)
                                  available by: inherited through CountableSets(ℤ) ≤ Sets
  ℚ                 union        

In [135]:
let A := [1, 2; 3, 4] in Mat₂(ℤ/5)

1:0: A := [1, 2; 3, 4] ∈ Mat₂(ℤ/5)


So the next cell **fails on purpose** — with a structured
`NoImplementation` capability gap naming the method, the receiver
category, the presentation, and the routes considered. Not a parse error,
not a type error, and not a silent lie. This failing cell is part of the
proof.


In [136]:
A.det()

LeanError: NoImplementation: 'det' is mathematically available here, but no registered route can execute it for this presentation.
  method:            det
  receiver category: MatrixElems(2, ℤ/5)
  presentation:      [1, 2; 3, 4] ∈ Mat₂(ℤ/5)
  semantic path:     declared directly on MatrixElems(2, ℤ/5)
  routes considered: 1
    - det for element of Mat(_, ℚ) → backend sage, op "mat_det_q", priority 0
This is a developer backlog item, not a narrowing of the mathematics: the method stays available on the category.